# 04 · Ekspor Data untuk Dashboard Web Story

Notebook ini **hanya membaca** hasil analisis yang sudah ada, lalu menulis satu berkas `data.js` untuk dashboard.
Tidak ada perhitungan ulang IRIO maupun LISA, dan tidak ada berkas analisis yang ditimpa.
Hasilnya disimpan di folder terpisah `Data/dashboard/`.

| Sumber | Dari notebook |
|---|---|
| A1, A3, B2 (`csv_output/`) | 03_IRIO_Simulator_Bali_v3 |
| F1_LISA_Klasifikasi | Notebook Tujuan 3 |
| indonesia_prov.geojson | 03_IRIO_Simulator_Bali_v3 (sudah diunduh sebelumnya) |

In [2]:
# Sel 1 — Hubungkan Drive dan tentukan lokasi (sama dengan notebook analisis)
import os, glob, json
import pandas as pd
import requests
from google.colab import drive, files
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/'
CSV_DIR = BASE + 'Output Simulasi Shock/'
F_GEO   = BASE + 'indonesia_prov.geojson'
OUT_DIR = BASE + 'dashboard/'          # folder baru, khusus dashboard
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
# Sel 2 — Cari berkas: di csv_output dulu, kalau tidak ada cari di seluruh folder Data
def cari(nama):
    lokasi = CSV_DIR + nama
    if os.path.exists(lokasi):
        return lokasi
    # glob.escape wajib karena nama folder memakai kurung siku, misalnya [01]
    hasil = glob.glob(glob.escape(BASE) + '**/' + nama, recursive=True)
    assert hasil, f'Tidak ditemukan di mana pun: {nama}'
    return max(hasil, key=os.path.getmtime)   # kalau ada beberapa, ambil yang terbaru

F_A1 = cari('A1_Dampak_per_Provinsi.csv')
F_A3 = cari('A3_Spillover_per_Sektor.csv')
F_B2 = cari('B2_Dekomposisi_Dampak.csv')
F_F1 = cari('F1_LISA_Klasifikasi.csv')

if not os.path.exists(F_GEO):   # seharusnya sudah ada dari notebook 03
    r = requests.get('https://raw.githubusercontent.com/ans-4175/peta-indonesia-geojson/'
                     'master/indonesia-prov.geojson', timeout=120)
    r.raise_for_status(); open(F_GEO, 'wb').write(r.content)

for f in [F_A1, F_A3, F_B2, F_F1, F_GEO]:
    print('Dipakai:', f.replace(BASE, 'Data/'))

Dipakai: Data/Output Simulasi Shock/A1_Dampak_per_Provinsi.csv
Dipakai: Data/Output Simulasi Shock/A3_Spillover_per_Sektor.csv
Dipakai: Data/Output Simulasi Shock/B2_Dekomposisi_Dampak.csv
Dipakai: Data/Output Analisis Spasial Overlay/F1_LISA_Klasifikasi.csv
Dipakai: Data/indonesia_prov.geojson


In [4]:
# Sel 3 — Baca data
a1 = pd.read_csv(F_A1, encoding='utf-8-sig')
a3 = pd.read_csv(F_A3, encoding='utf-8-sig')
b2 = pd.read_csv(F_B2, encoding='utf-8-sig')
f1 = pd.read_csv(F_F1, encoding='utf-8-sig')
print(f'A1: {len(a1)} provinsi | A3: {len(a3)} sektor | B2: {len(b2)} baris | F1: {len(f1)} provinsi')

A1: 34 provinsi | A3: 17 sektor | B2: 5 baris | F1: 33 provinsi


In [5]:
# Sel 4 — Cek konsistensi: angka ini harus sama dengan naskah skripsi
total  = b2.set_index('Komponen').loc['Total dampak']
langs  = b2.set_index('Komponen').loc['Efek langsung']
spill  = b2.set_index('Komponen').loc['Spillover interregional']
luar   = a1[a1['Provinsi'] != 'Bali']

print(f"Shock permintaan akhir : Rp{-langs['Dampak_triliun']:,.2f} T      (naskah: 84,39)")
print(f"Dampak output nasional : Rp{-total['Dampak_triliun']:,.2f} T     (naskah: 142,65)")
print(f"Multiplier             : {total['Multiplier_komponen']:.4f}       (naskah: 1,6903)")
print(f"Porsi Bali             : {a1.loc[a1['Provinsi'] == 'Bali', 'Pct_nasional'].item():.2f} %      (naskah: 78,81)")
print(f"Porsi spillover        : {spill['Porsi_pct']:.2f} %      (naskah: 21,19)")
print('Spillover terbesar     :', luar.loc[luar['Dx_juta'].idxmin(), 'Provinsi'], '   (naskah: Jawa Timur)')
print('Intensitas tertinggi   :', luar.loc[luar['Intensitas_pct'].idxmax(), 'Provinsi'], '  (naskah: Sumatera Selatan)')
print('Klaster HH             :', sorted(f1.loc[f1['Kuadran_Final'] == 'HH', 'Provinsi']))
print('Klaster LH             :', sorted(f1.loc[f1['Kuadran_Final'] == 'LH', 'Provinsi']))

Shock permintaan akhir : Rp84.39 T      (naskah: 84,39)
Dampak output nasional : Rp142.65 T     (naskah: 142,65)
Multiplier             : 1.6903       (naskah: 1,6903)
Porsi Bali             : 78.81 %      (naskah: 78,81)
Porsi spillover        : 21.19 %      (naskah: 21,19)
Spillover terbesar     : Jawa Timur    (naskah: Jawa Timur)
Intensitas tertinggi   : Sumatera Selatan   (naskah: Sumatera Selatan)
Klaster HH             : ['Jawa Timur', 'Nusa Tenggara Barat', 'Nusa Tenggara Timur']
Klaster LH             : ['DI Yogyakarta', 'Kep. Bangka Belitung']


In [6]:
# Sel 5 — Gabungkan data provinsi, tempelkan ke GeoJSON, tulis data.js
NAMA_GEOJSON = {'Aceh': 'DI. ACEH', 'Kep. Bangka Belitung': 'BANGKA BELITUNG',
                'Kep. Riau': 'KEPULAUAN RIAU', 'DI Yogyakarta': 'DAERAH ISTIMEWA YOGYAKARTA',
                'Nusa Tenggara Barat': 'NUSATENGGARA BARAT'}   # disalin dari notebook 03

prov = a1.merge(f1[['Provinsi', 'Local_I', 'p_value', 'Kuadran_Final']],
                on='Provinsi', how='left')
prov['Kuadran_Final'] = prov['Kuadran_Final'].fillna('Sumber shock')   # Bali tidak ikut LISA
prov = prov.astype(object).where(prov.notna(), None)
baris = prov.set_index('Provinsi').to_dict('index')

geo = json.load(open(F_GEO, encoding='utf-8'))
ke_csv = {NAMA_GEOJSON.get(n, n.upper()): n for n in prov['Provinsi']}
for fitur in geo['features']:
    nama = ke_csv[fitur['properties']['Propinsi']]
    fitur['properties'] = {'Provinsi': nama, **baris[nama]}

DATA = {'geo': geo,
        'dekomposisi': b2.to_dict('records'),
        'sektor': a3.to_dict('records')}
F_JS = OUT_DIR + 'data.js'
with open(F_JS, 'w', encoding='utf-8') as f:
    f.write('const DATA = ' + json.dumps(DATA, ensure_ascii=False) + ';')

print('Selesai:', len(geo['features']), 'provinsi,', len(DATA['sektor']), 'sektor,',
      len(DATA['dekomposisi']), 'baris dekomposisi')
print(f'Tersimpan di Data/dashboard/data.js ({os.path.getsize(F_JS) / 1e3:,.0f} KB)')

Selesai: 34 provinsi, 17 sektor, 5 baris dekomposisi
Tersimpan di Data/dashboard/data.js (489 KB)


In [7]:
# Sel 6 — Unduh ke komputer, lalu taruh di folder data/ proyek VS Code
files.download(F_JS)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>